# YYW affine-Gaussian batch fitting

Fit the previously analyzed `55A_R0`, `55B_R0`, and `55C_R0` curves with the heterogeneous random-line model. Each curve starts from its own earlier fitted `mean_k`, `r_sigma_k`, `k_H_over_k`, and `b`. The previous skewed maximum-entropy distribution is replaced by the simplest positive `gaussian_radial` distribution. An incompressible uniaxial real-space stretch is added as one new fitted parameter, initially `stretch_ratio = 1.2`.

At every model trial, the isotropic line spectrum is calculated over the full stretched-Q support, powder averaged by Gauss--Legendre angular quadrature, and then compared with the observation. The log-stretch derivative series is not used.


In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np

import cf_tools as cf

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "axes.grid": True, "grid.alpha": 0.25})

def load_parameter_csv(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return {row["name"]: float(row["value"]) for row in csv.DictReader(handle)}

def load_profile_csv(path, label):
    table = np.loadtxt(path, delimiter=",", skiprows=1)
    table = np.atleast_2d(table)
    return cf.RadialProfile(label=label, q=table[:, 0], intensity=table[:, 1], error=table[:, 2], count=table[:, 3].astype(int))


## Inputs and numerical controls

In [ ]:
PREVIOUS_OUTPUT_DIR = Path("output/yyw")
OUTPUT_DIR = Path("output/yyw/aniso")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    tag: {
        "observation": PREVIOUS_OUTPUT_DIR/f"{tag}_stitched_observation.csv",
        "parameters": PREVIOUS_OUTPUT_DIR/f"{tag}_first_fit_parameters.csv",
    }
    for tag in ("55A_R0", "55B_R0", "55C_R0")
}
for tag, files in SAMPLES.items():
    for kind, path in files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing previous {kind} for {tag}: {path}")

FIT_Q_BOUNDS = (1.5e-3, 4.0e-1)
LOWQ_ANCHOR_BOUNDS = (1.5e-3, 8.0e-3)
HIGHQ_ANCHOR_BOUNDS = (8.0e-2, 4.0e-1)
LOG_ERROR_FLOOR = 0.04
ANCHOR_WEIGHT = 1.0
STRETCH_INITIAL = 1.2

fit_model_settings = {
    "k_distribution": "gaussian_radial",
    "k_sampling": "quadrature",
    "regression_loss": "relative",
    "num_modes_k": 2**10,
    "Nr": 3000,
    "Nr_small": 600,
    "r_min_factor": 1e-3,
    "r_split_factor": 5.0,
    "r_max_factor": 250.0,
    "N_samp_U": 2**12,
    "N_samp_st": 2**7,
    "NQ": 220,
    "powder_n_mu": 64,
    "progress": False,
}
distribution_parameter_set = {"r_sigma_k": {"initial": 0.27, "bounds": (0.06, 0.90)}}
RUN_FITS = True
MAX_NFEV = 80


## Load previous fits and construct Gaussian--affine initial models

In [ ]:
profiles = {}
previous_parameters = {}
fit_initials = {}
fit_bounds_by_sample = {}
anchors = {}
initial_models = {}

for tag, files in SAMPLES.items():
    profile = load_profile_csv(files["observation"], tag)
    previous = load_parameter_csv(files["parameters"])
    profiles[tag] = profile
    previous_parameters[tag] = previous
    initial = {
        "mean_k": previous["mean_k"],
        "r_sigma_k": previous["r_sigma_k"],
        "k_H_over_k": previous["k_H_over_k"],
        "b": previous["b"],
        "stretch_ratio": STRETCH_INITIAL,
    }
    if tag == "55A_R0":
        initial["r_sigma_k"] = min(initial["r_sigma_k"], 0.24)
        r_sigma_bounds = (0.06, 0.25)
    else:
        r_sigma_bounds = distribution_parameter_set["r_sigma_k"]["bounds"]
    fit_initials[tag] = initial
    fit_bounds_by_sample[tag] = {
        "mean_k": (max(0.005, 0.45*initial["mean_k"]), min(0.5, 2.2*initial["mean_k"])),
        "r_sigma_k": r_sigma_bounds,
        "k_H_over_k": (max(1e-4, 0.25*initial["k_H_over_k"]), min(0.5, 4.0*initial["k_H_over_k"])),
        "b": (-4.0, 2.0),
        "stretch_ratio": (0.85, 1.65),
    }
    anchors[tag] = {
        "low": cf.fit_lowq_dab_anchor(profile, LOWQ_ANCHOR_BOUNDS, background_max=0.0),
        "high": cf.fit_highq_line_anchor(profile, HIGHQ_ANCHOR_BOUNDS, background_max=0.0),
    }
    initial_models[tag] = cf.evaluate_heterogeneous_line_guess(
        profile, q_bounds=FIT_Q_BOUNDS, parameters=initial,
        model_settings=fit_model_settings,
        distribution_parameter_set=distribution_parameter_set,
        log_error_floor=LOG_ERROR_FLOOR, affine_stretch=True,
    )
    print(f"[{tag}] previous -> Gaussian-affine initial")
    for name, value in initial.items():
        print(f"  {name:16s} {value:.8g}")
    print(f"  initial cost     {initial_models[tag].cost:.8g}")


In [ ]:
fig, axes = plt.subplots(len(SAMPLES), 2, figsize=(11, 10), sharex="col")
for row, tag in enumerate(SAMPLES):
    model = initial_models[tag]
    axes[row, 0].errorbar(model.q, model.intensity, yerr=model.error, fmt="o", ms=2.5, lw=0.5, color="k", markerfacecolor="none", label=tag)
    axes[row, 0].plot(model.q, model.model, color="C3", lw=1.4, label="Gaussian-affine initial")
    axes[row, 0].set(xscale="log", yscale="log", ylabel=r"$I(Q)$")
    axes[row, 0].legend(fontsize=8)
    axes[row, 1].semilogx(model.q, model.model/model.intensity, color="C3")
    axes[row, 1].axhline(1.0, color="k", lw=0.8)
    axes[row, 1].set(ylabel=r"$I_{initial}/I_{obs}$")
axes[-1, 0].set_xlabel(r"$Q$ ($\AA^{-1}$)")
axes[-1, 1].set_xlabel(r"$Q$ ($\AA^{-1}$)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"yyw_gaussian_affine_initial_models.png")
plt.show()


## Fit every curve

The earlier physical parameters are starting values, not fixed constraints. The optimizer varies `mean_k`, `r_sigma_k`, `k_H_over_k`, `b`, and the new `stretch_ratio`; the overall intensity scale is solved analytically at every trial.


In [ ]:
fit_results = {}
if RUN_FITS:
    for tag in SAMPLES:
        print(f"[fit] {tag}")
        result = cf.fit_heterogeneous_line_least_squares(
            profiles[tag], q_bounds=FIT_Q_BOUNDS, initial=fit_initials[tag],
            lowq_kappa_anchor=anchors[tag]["low"].parameters["kappa"],
            highq_coefficient_anchor=anchors[tag]["high"].parameters["coefficient"],
            bounds=fit_bounds_by_sample[tag],
            model_settings=fit_model_settings,
            distribution_parameter_set=distribution_parameter_set,
            max_nfev=MAX_NFEV, anchor_weight=ANCHOR_WEIGHT,
            log_error_floor=LOG_ERROR_FLOOR, affine_stretch=True,
        )
        fit_results[tag] = result
        rms_relative = float(np.sqrt(np.mean((result.model/result.intensity-1.0)**2)))
        result.parameters["rms_relative"] = rms_relative
        print(f"  success={result.success}; nfev={result.nfev}; cost={result.cost:.8g}; RMSrel={rms_relative:.6g}")
        for name in ("mean_k", "r_sigma_k", "k_H_over_k", "b", "stretch_ratio"):
            print(f"  {name:16s} {result.parameters[name]:.8g}")
        with (OUTPUT_DIR/f"{tag}_gaussian_affine_fit_parameters.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle); writer.writerow(["name", "value"]); writer.writerows(result.parameters.items())
        np.savetxt(OUTPUT_DIR/f"{tag}_gaussian_affine_fit_curve.csv", np.column_stack([result.q, result.intensity, result.error, result.model, result.model/result.intensity]), delimiter=",", header="Q,I_obs,err,I_fit,I_fit_over_I_obs", comments="")
else:
    print("[status] fits not run; inspect initial models and set RUN_FITS=True")


In [ ]:
if RUN_FITS:
    summary_names = ["mean_k", "r_sigma_k", "k_H_over_k", "b", "stretch_ratio", "scale", "rms_relative"]
    with (OUTPUT_DIR/"yyw_gaussian_affine_fit_summary.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle); writer.writerow(["sample", *summary_names])
        for tag, result in fit_results.items(): writer.writerow([tag, *[result.parameters[name] for name in summary_names]])
    fig, axes = plt.subplots(len(SAMPLES), 2, figsize=(11, 10), sharex="col")
    for row, tag in enumerate(SAMPLES):
        result = fit_results[tag]
        axes[row, 0].errorbar(result.q, result.intensity, yerr=result.error, fmt="o", ms=2.5, lw=0.5, color="k", markerfacecolor="none", label=tag)
        axes[row, 0].plot(result.q, result.model, color="C3", lw=1.5, label=f"Gaussian affine, stretch={result.parameters['stretch_ratio']:.3f}")
        axes[row, 0].set(xscale="log", yscale="log", ylabel=r"$I(Q)$")
        axes[row, 0].legend(fontsize=8)
        axes[row, 1].semilogx(result.q, result.model/result.intensity, "o-", ms=2.5, lw=0.9)
        axes[row, 1].axhline(1.0, color="k", lw=0.8)
        axes[row, 1].set(ylabel=r"$I_{fit}/I_{obs}$")
    axes[-1, 0].set_xlabel(r"$Q$ ($\AA^{-1}$)")
    axes[-1, 1].set_xlabel(r"$Q$ ($\AA^{-1}$)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/"yyw_gaussian_affine_fits.png")
    plt.show()
